# Pharma Registry API Explorer

This notebook explores the major pharmaceutical drug registration and clinical trial APIs:

1. **FDA openFDA API** - US approved drugs
2. **Health Canada Drug Product Database API** - Canadian approved drugs
3. **ClinicalTrials.gov API v2.0** - Global clinical trials
4. **EMA Data Downloads** - EU centrally authorized medicines

---

In [ ]:
# Required imports
import requests
import pandas as pd
import json
from datetime import datetime
from typing import Optional, Dict, List, Any

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

---
## 1. FDA openFDA API

The openFDA API provides access to FDA drug data including:
- Drug product labeling
- Adverse events (FAERS)
- NDC Directory
- Drugs@FDA (approval information)

**Base URL:** `https://api.fda.gov/drug/`

**Documentation:** https://open.fda.gov/apis/drug/

In [ ]:
class OpenFDAClient:
    """Client for the FDA openFDA API."""
    
    BASE_URL = "https://api.fda.gov/drug"
    
    def __init__(self, api_key: Optional[str] = None):
        """Initialize client. API key is optional but increases rate limits."""
        self.api_key = api_key
        self.session = requests.Session()
    
    def _make_request(self, endpoint: str, params: Dict[str, Any]) -> Dict:
        """Make API request with error handling."""
        if self.api_key:
            params['api_key'] = self.api_key
        
        url = f"{self.BASE_URL}/{endpoint}.json"
        response = self.session.get(url, params=params)
        response.raise_for_status()
        return response.json()
    
    def search_drugsfda(self, search: str, limit: int = 10) -> Dict:
        """Search Drugs@FDA database for approved drug products."""
        params = {'search': search, 'limit': limit}
        return self._make_request('drugsfda', params)
    
    def search_label(self, search: str, limit: int = 10) -> Dict:
        """Search drug labeling (package inserts)."""
        params = {'search': search, 'limit': limit}
        return self._make_request('label', params)
    
    def search_ndc(self, search: str, limit: int = 10) -> Dict:
        """Search NDC Directory."""
        params = {'search': search, 'limit': limit}
        return self._make_request('ndc', params)
    
    def get_adverse_events(self, search: str, limit: int = 10) -> Dict:
        """Search FDA Adverse Event Reporting System (FAERS)."""
        params = {'search': search, 'limit': limit}
        return self._make_request('event', params)
    
    def count_by_field(self, endpoint: str, search: str, count_field: str, limit: int = 100) -> Dict:
        """Get counts grouped by a field."""
        params = {'search': search, 'count': count_field, 'limit': limit}
        return self._make_request(endpoint, params)

# Initialize client
fda_client = OpenFDAClient()

### 1.1 Search Drugs@FDA for Approved Products

In [ ]:
# Search for recently approved drugs (2024)
try:
    result = fda_client.search_drugsfda(
        search='submissions.submission_status_date:[20240101 TO 20241231]',
        limit=5
    )
    
    print(f"Total matching records: {result['meta']['results']['total']}")
    print(f"\nFirst {len(result['results'])} results:\n")
    
    for drug in result['results']:
        sponsor = drug.get('sponsor_name', 'N/A')
        products = drug.get('products', [{}])
        brand = products[0].get('brand_name', 'N/A') if products else 'N/A'
        active = products[0].get('active_ingredients', [{}])[0].get('name', 'N/A') if products else 'N/A'
        app_num = drug.get('application_number', 'N/A')
        
        print(f"  • {brand} ({active})")
        print(f"    Sponsor: {sponsor}")
        print(f"    Application: {app_num}\n")

except Exception as e:
    print(f"Error: {e}")

### 1.2 Search Drug Labels by Active Ingredient

In [ ]:
# Search for drug labels containing "metformin"
try:
    result = fda_client.search_label(
        search='openfda.generic_name:metformin',
        limit=5
    )
    
    print(f"Total metformin products: {result['meta']['results']['total']}\n")
    
    records = []
    for label in result['results']:
        openfda = label.get('openfda', {})
        records.append({
            'brand_name': openfda.get('brand_name', ['N/A'])[0],
            'generic_name': openfda.get('generic_name', ['N/A'])[0],
            'manufacturer': openfda.get('manufacturer_name', ['N/A'])[0],
            'route': openfda.get('route', ['N/A'])[0],
            'product_type': openfda.get('product_type', ['N/A'])[0]
        })
    
    df = pd.DataFrame(records)
    display(df)

except Exception as e:
    print(f"Error: {e}")

### 1.3 Count Drugs by Manufacturer

In [ ]:
# Get top manufacturers by number of drug labels
try:
    result = fda_client.count_by_field(
        endpoint='label',
        search='_exists_:openfda.manufacturer_name',
        count_field='openfda.manufacturer_name.exact',
        limit=15
    )
    
    df = pd.DataFrame(result['results'])
    df.columns = ['manufacturer', 'count']
    
    print("Top 15 Manufacturers by Drug Label Count:\n")
    display(df)

except Exception as e:
    print(f"Error: {e}")

---
## 2. Health Canada Drug Product Database (DPD) API

Health Canada provides a comprehensive REST API for the Drug Product Database.

**Base URL:** `https://health-products.canada.ca/api/drug/`

**Documentation:** https://health-products.canada.ca/api/documentation/dpd-documentation-en.html

In [ ]:
class HealthCanadaDPDClient:
    """Client for Health Canada Drug Product Database API."""
    
    BASE_URL = "https://health-products.canada.ca/api/drug"
    
    def __init__(self):
        self.session = requests.Session()
    
    def _make_request(self, endpoint: str, params: Optional[Dict] = None) -> Any:
        """Make API request."""
        url = f"{self.BASE_URL}/{endpoint}"
        response = self.session.get(url, params=params)
        response.raise_for_status()
        return response.json()
    
    def get_drug_product(self, drug_code: int) -> Dict:
        """Get drug product by drug code."""
        return self._make_request('drugproduct', {'id': drug_code})
    
    def search_brand_name(self, brand_name: str) -> List[Dict]:
        """Search by brand name."""
        return self._make_request('brand', {'lang': 'en', 'brandname': brand_name})
    
    def search_ingredient(self, ingredient: str) -> List[Dict]:
        """Search by active ingredient."""
        return self._make_request('activeingredient', {'lang': 'en', 'ingredient': ingredient})
    
    def search_company(self, company_name: str) -> List[Dict]:
        """Search by company name."""
        return self._make_request('company', {'lang': 'en', 'companyname': company_name})
    
    def get_all_drug_products(self, status: str = 'approved') -> List[Dict]:
        """Get all drug products by status (approved, marketed, cancelled, dormant)."""
        return self._make_request(f'drugproduct/?status={status}')
    
    def get_route(self, drug_code: int) -> List[Dict]:
        """Get route of administration for a drug."""
        return self._make_request('route', {'id': drug_code, 'lang': 'en'})
    
    def get_form(self, drug_code: int) -> List[Dict]:
        """Get pharmaceutical form for a drug."""
        return self._make_request('form', {'id': drug_code, 'lang': 'en'})

# Initialize client
hc_client = HealthCanadaDPDClient()

### 2.1 Search by Brand Name

In [ ]:
# Search for drugs with "Tylenol" in brand name
try:
    results = hc_client.search_brand_name('Tylenol')
    
    print(f"Found {len(results)} products with 'Tylenol' in brand name:\n")
    
    records = []
    for item in results[:10]:  # First 10
        records.append({
            'drug_code': item.get('drug_code'),
            'brand_name': item.get('brand_name'),
            'company_name': item.get('company_name'),
            'status': item.get('status'),
            'class_name': item.get('class_name')
        })
    
    df = pd.DataFrame(records)
    display(df)

except Exception as e:
    print(f"Error: {e}")

### 2.2 Search by Active Ingredient

In [ ]:
# Search for products containing ibuprofen
try:
    results = hc_client.search_ingredient('ibuprofen')
    
    print(f"Found {len(results)} products containing ibuprofen:\n")
    
    records = []
    for item in results[:10]:  # First 10
        records.append({
            'drug_code': item.get('drug_code'),
            'ingredient': item.get('ingredient'),
            'strength': item.get('strength'),
            'strength_unit': item.get('strength_unit'),
            'dosage_value': item.get('dosage_value'),
            'dosage_unit': item.get('dosage_unit')
        })
    
    df = pd.DataFrame(records)
    display(df)

except Exception as e:
    print(f"Error: {e}")

### 2.3 Search by Company

In [ ]:
# Search for products by a pharmaceutical company
try:
    results = hc_client.search_company('Pfizer')
    
    print(f"Found {len(results)} products from companies matching 'Pfizer':\n")
    
    records = []
    for item in results[:10]:  # First 10
        records.append({
            'drug_code': item.get('drug_code'),
            'company_name': item.get('company_name'),
            'company_code': item.get('company_code'),
            'company_type': item.get('company_type'),
            'city': item.get('city'),
            'country': item.get('country')
        })
    
    df = pd.DataFrame(records)
    display(df)

except Exception as e:
    print(f"Error: {e}")

---
## 3. ClinicalTrials.gov API v2.0

The world's largest clinical trial registry with ~500,000 studies.

**Base URL:** `https://clinicaltrials.gov/api/v2/`

**Documentation:** https://clinicaltrials.gov/data-api/api

In [ ]:
class ClinicalTrialsClient:
    """Client for ClinicalTrials.gov API v2.0."""
    
    BASE_URL = "https://clinicaltrials.gov/api/v2"
    
    def __init__(self):
        self.session = requests.Session()
    
    def _make_request(self, endpoint: str, params: Optional[Dict] = None) -> Dict:
        """Make API request."""
        url = f"{self.BASE_URL}/{endpoint}"
        response = self.session.get(url, params=params)
        response.raise_for_status()
        return response.json()
    
    def search_studies(self, 
                       query_term: Optional[str] = None,
                       query_cond: Optional[str] = None,
                       query_intr: Optional[str] = None,
                       filter_overall_status: Optional[str] = None,
                       filter_geo: Optional[str] = None,
                       page_size: int = 10,
                       page_token: Optional[str] = None,
                       fields: Optional[List[str]] = None) -> Dict:
        """Search clinical trials.
        
        Args:
            query_term: General search term
            query_cond: Condition/disease search
            query_intr: Intervention/treatment search
            filter_overall_status: Status filter (e.g., 'RECRUITING')
            filter_geo: Geographic filter (e.g., 'distance(39.0035,-77.1013,50mi)')
            page_size: Number of results per page (max 1000)
            page_token: Token for pagination
            fields: Specific fields to return
        """
        params = {'pageSize': page_size}
        
        if query_term:
            params['query.term'] = query_term
        if query_cond:
            params['query.cond'] = query_cond
        if query_intr:
            params['query.intr'] = query_intr
        if filter_overall_status:
            params['filter.overallStatus'] = filter_overall_status
        if filter_geo:
            params['filter.geo'] = filter_geo
        if page_token:
            params['pageToken'] = page_token
        if fields:
            params['fields'] = '|'.join(fields)
        
        return self._make_request('studies', params)
    
    def get_study(self, nct_id: str, fields: Optional[List[str]] = None) -> Dict:
        """Get a specific study by NCT ID."""
        params = {}
        if fields:
            params['fields'] = '|'.join(fields)
        return self._make_request(f'studies/{nct_id}', params)
    
    def get_study_sizes(self) -> Dict:
        """Get statistics about study field sizes."""
        return self._make_request('stats/size')
    
    def get_field_values(self, field: str) -> Dict:
        """Get possible values for a specific field."""
        return self._make_request(f'stats/fieldValues/{field}')

# Initialize client
ct_client = ClinicalTrialsClient()

### 3.1 Search for Recruiting Studies by Condition

In [ ]:
# Search for recruiting diabetes trials
try:
    result = ct_client.search_studies(
        query_cond='diabetes',
        filter_overall_status='RECRUITING',
        page_size=10,
        fields=['NCTId', 'BriefTitle', 'OverallStatus', 'Phase', 'EnrollmentCount', 
                'StartDate', 'LeadSponsorName', 'LocationCountry']
    )
    
    print(f"Total recruiting diabetes studies: {result.get('totalCount', 'N/A')}\n")
    
    records = []
    for study in result.get('studies', []):
        protocol = study.get('protocolSection', {})
        id_module = protocol.get('identificationModule', {})
        status_module = protocol.get('statusModule', {})
        design_module = protocol.get('designModule', {})
        sponsor_module = protocol.get('sponsorCollaboratorsModule', {})
        contacts_module = protocol.get('contactsLocationsModule', {})
        
        records.append({
            'nct_id': id_module.get('nctId'),
            'title': id_module.get('briefTitle', '')[:80] + '...' if len(id_module.get('briefTitle', '')) > 80 else id_module.get('briefTitle'),
            'status': status_module.get('overallStatus'),
            'phase': ', '.join(design_module.get('phases', ['N/A'])),
            'enrollment': design_module.get('enrollmentInfo', {}).get('count'),
            'sponsor': sponsor_module.get('leadSponsor', {}).get('name'),
            'countries': ', '.join(contacts_module.get('locations', [{}])[0].get('country', 'N/A') if contacts_module.get('locations') else ['N/A'])
        })
    
    df = pd.DataFrame(records)
    display(df)

except Exception as e:
    print(f"Error: {e}")

### 3.2 Search by Intervention/Drug

In [ ]:
# Search for trials involving a specific drug
try:
    result = ct_client.search_studies(
        query_intr='pembrolizumab',
        page_size=10,
        fields=['NCTId', 'BriefTitle', 'OverallStatus', 'Phase', 'Condition', 'LeadSponsorName']
    )
    
    print(f"Total pembrolizumab studies: {result.get('totalCount', 'N/A')}\n")
    
    records = []
    for study in result.get('studies', []):
        protocol = study.get('protocolSection', {})
        id_module = protocol.get('identificationModule', {})
        status_module = protocol.get('statusModule', {})
        design_module = protocol.get('designModule', {})
        cond_module = protocol.get('conditionsModule', {})
        sponsor_module = protocol.get('sponsorCollaboratorsModule', {})
        
        conditions = cond_module.get('conditions', [])
        
        records.append({
            'nct_id': id_module.get('nctId'),
            'title': id_module.get('briefTitle', '')[:60] + '...' if len(id_module.get('briefTitle', '')) > 60 else id_module.get('briefTitle'),
            'status': status_module.get('overallStatus'),
            'phase': ', '.join(design_module.get('phases', ['N/A'])),
            'conditions': ', '.join(conditions[:2]) + ('...' if len(conditions) > 2 else ''),
            'sponsor': sponsor_module.get('leadSponsor', {}).get('name')
        })
    
    df = pd.DataFrame(records)
    display(df)

except Exception as e:
    print(f"Error: {e}")

### 3.3 Get Details for a Specific Trial

In [ ]:
# Get detailed information about a specific trial
try:
    # Example: A well-known cancer trial
    study = ct_client.get_study('NCT02578680')  # KEYNOTE-024 (pembrolizumab)
    
    protocol = study.get('protocolSection', {})
    
    # Extract key information
    id_module = protocol.get('identificationModule', {})
    status_module = protocol.get('statusModule', {})
    sponsor_module = protocol.get('sponsorCollaboratorsModule', {})
    desc_module = protocol.get('descriptionModule', {})
    design_module = protocol.get('designModule', {})
    eligibility_module = protocol.get('eligibilityModule', {})
    
    print(f"=== {id_module.get('nctId')} ===")
    print(f"\nTitle: {id_module.get('officialTitle', id_module.get('briefTitle'))}")
    print(f"\nStatus: {status_module.get('overallStatus')}")
    print(f"Phase: {', '.join(design_module.get('phases', ['N/A']))}")
    print(f"Study Type: {design_module.get('studyType')}")
    print(f"\nSponsor: {sponsor_module.get('leadSponsor', {}).get('name')}")
    print(f"\nBrief Summary:")
    print(desc_module.get('briefSummary', 'N/A')[:500] + '...')
    print(f"\nEligibility:")
    print(f"  Age: {eligibility_module.get('minimumAge', 'N/A')} - {eligibility_module.get('maximumAge', 'N/A')}")
    print(f"  Sex: {eligibility_module.get('sex', 'N/A')}")
    print(f"  Healthy Volunteers: {eligibility_module.get('healthyVolunteers', 'N/A')}")

except Exception as e:
    print(f"Error: {e}")

### 3.4 Get Database Statistics

In [ ]:
# Get overall database statistics
try:
    # Get possible values for overall status field
    status_values = ct_client.get_field_values('OverallStatus')
    
    print("Study Status Distribution:\n")
    for item in status_values.get('fieldValues', []):
        print(f"  {item.get('value')}: {item.get('count'):,}")
    
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Get phase distribution
try:
    phase_values = ct_client.get_field_values('Phase')
    
    print("Study Phase Distribution:\n")
    for item in phase_values.get('fieldValues', []):
        print(f"  {item.get('value')}: {item.get('count'):,}")

except Exception as e:
    print(f"Error: {e}")

---
## 4. EMA Medicine Data (Download)

EMA provides downloadable data files in Excel/JSON format.

**Download Page:** https://www.ema.europa.eu/en/medicines/download-medicine-data

Let's fetch the publicly available medicine data.

In [ ]:
class EMADataClient:
    """Client for EMA downloadable data."""
    
    # EMA provides Excel files that can be downloaded directly
    MEDICINES_URL = "https://www.ema.europa.eu/sites/default/files/Medicines_output_european_public_assessment_reports.xlsx"
    
    def __init__(self):
        self.session = requests.Session()
    
    def get_medicines_data(self) -> pd.DataFrame:
        """Download and parse EMA medicines data."""
        try:
            # Try to read directly from URL
            df = pd.read_excel(self.MEDICINES_URL)
            return df
        except Exception as e:
            print(f"Could not fetch EMA data directly: {e}")
            print("\nYou can manually download from:")
            print("https://www.ema.europa.eu/en/medicines/download-medicine-data")
            return pd.DataFrame()

# Initialize client
ema_client = EMADataClient()

In [ ]:
# Try to fetch EMA medicines data
try:
    ema_df = ema_client.get_medicines_data()
    
    if not ema_df.empty:
        print(f"EMA Medicines Database: {len(ema_df)} records\n")
        print("Columns available:")
        for col in ema_df.columns:
            print(f"  • {col}")
        
        print(f"\nFirst 5 records:")
        display(ema_df.head())
    else:
        print("EMA data not available via direct download.")
        print("Manual download required from EMA website.")

except Exception as e:
    print(f"Error: {e}")

---
## 5. Utility Functions for Cross-Registry Analysis

In [ ]:
def search_drug_across_registries(drug_name: str) -> Dict[str, Any]:
    """Search for a drug across multiple registries."""
    
    results = {}
    
    # FDA
    print(f"Searching FDA for '{drug_name}'...")
    try:
        fda_result = fda_client.search_label(f'openfda.generic_name:{drug_name}', limit=5)
        results['FDA'] = {
            'total': fda_result['meta']['results']['total'],
            'sample': len(fda_result['results'])
        }
    except Exception as e:
        results['FDA'] = {'error': str(e)}
    
    # Health Canada
    print(f"Searching Health Canada for '{drug_name}'...")
    try:
        hc_result = hc_client.search_ingredient(drug_name)
        results['Health Canada'] = {
            'total': len(hc_result),
            'sample': min(5, len(hc_result))
        }
    except Exception as e:
        results['Health Canada'] = {'error': str(e)}
    
    # ClinicalTrials.gov
    print(f"Searching ClinicalTrials.gov for '{drug_name}'...")
    try:
        ct_result = ct_client.search_studies(query_intr=drug_name, page_size=1)
        results['ClinicalTrials.gov'] = {
            'total': ct_result.get('totalCount', 0),
            'sample': len(ct_result.get('studies', []))
        }
    except Exception as e:
        results['ClinicalTrials.gov'] = {'error': str(e)}
    
    return results

In [ ]:
# Example: Search for a drug across registries
drug_to_search = "metformin"

print(f"\n=== Cross-Registry Search: {drug_to_search} ===\n")
cross_results = search_drug_across_registries(drug_to_search)

print("\nResults Summary:")
print("-" * 50)
for registry, data in cross_results.items():
    if 'error' in data:
        print(f"{registry}: Error - {data['error']}")
    else:
        print(f"{registry}: {data['total']:,} total records")

---
## 6. Summary and API Comparison

| Feature | FDA openFDA | Health Canada DPD | ClinicalTrials.gov |
|---------|------------|-------------------|--------------------|
| **Base URL** | api.fda.gov | health-products.canada.ca/api | clinicaltrials.gov/api/v2 |
| **Auth Required** | Optional (rate limits) | No | No |
| **Rate Limits** | 240/min (no key), 120K/day (with key) | Not specified | Not specified |
| **Response Format** | JSON | JSON | JSON |
| **Pagination** | skip/limit | Varies | pageToken |
| **Bulk Download** | Yes (ZIP files) | Yes (data extracts) | Yes (bulk JSON) |
| **Documentation** | Excellent | Good | Excellent |

In [ ]:
# Print API endpoints summary
print("=" * 60)
print("PHARMA API QUICK REFERENCE")
print("=" * 60)

apis = [
    {
        'name': 'FDA openFDA',
        'base_url': 'https://api.fda.gov/drug/',
        'endpoints': ['drugsfda', 'label', 'ndc', 'event'],
        'docs': 'https://open.fda.gov/apis/drug/'
    },
    {
        'name': 'Health Canada DPD',
        'base_url': 'https://health-products.canada.ca/api/drug/',
        'endpoints': ['drugproduct', 'brand', 'activeingredient', 'company', 'route', 'form'],
        'docs': 'https://health-products.canada.ca/api/documentation/dpd-documentation-en.html'
    },
    {
        'name': 'ClinicalTrials.gov v2',
        'base_url': 'https://clinicaltrials.gov/api/v2/',
        'endpoints': ['studies', 'studies/{nctId}', 'stats/size', 'stats/fieldValues/{field}'],
        'docs': 'https://clinicaltrials.gov/data-api/api'
    }
]

for api in apis:
    print(f"\n{api['name']}")
    print(f"  Base URL: {api['base_url']}")
    print(f"  Endpoints: {', '.join(api['endpoints'])}")
    print(f"  Docs: {api['docs']}")

---

## Next Steps

1. **Get an FDA API key** for higher rate limits: https://open.fda.gov/apis/authentication/
2. **Explore bulk downloads** for large-scale analysis:
   - FDA: https://open.fda.gov/apis/downloads/
   - ClinicalTrials.gov: https://clinicaltrials.gov/data-api/about-api/csv-download
   - Health Canada: https://open.canada.ca/data/en/dataset/bf55e42a-63cb-4556-bfd8-44f26e5a36fe
3. **Combine with EMA data** for global coverage
4. **Build monitoring dashboards** for new approvals and trial updates